# CI360: load the raw H&M data into MySQL

Creates `hm_articles`, `hm_customers` and `hm_transactions` and loads the CSVs. Credentials come from `api/.env`.

**Warning:** the create cells start with `DROP TABLE IF EXISTS` and delete any data already loaded.

If the transactions load is interrupted, run `python resume_transactions_load.py` instead of re-running the load cell. After loading, continue with the pipeline in the README (`duckdb_metrics.py`, `setup_relationships.sql`, `score_churn.py`, `build_crm_customers.py`).

In [1]:
import csv

import pymysql

In [2]:
from db_config import get_connection  # credentials come from api/.env

conn = get_connection(autocommit=True)

cur = conn.cursor(pymysql.cursors.DictCursor)

In [12]:
cur.execute("DROP TABLE IF EXISTS hm_articles")
cur.execute("""
CREATE TABLE hm_articles (
    article_id BIGINT PRIMARY KEY,
    prod_name VARCHAR(255),
    product_type_name VARCHAR(255),
    product_group_name VARCHAR(255),
    department_name VARCHAR(255),
    index_name VARCHAR(255),
    colour_group_name VARCHAR(255),
    garment_group_name VARCHAR(255),
    detail_desc TEXT
)ENGINE=InnoDB;
""")

0

In [3]:
cur.execute("DROP TABLE IF EXISTS hm_customers;")
cur.execute("""
CREATE TABLE hm_customers (
    customer_id VARCHAR(64) PRIMARY KEY,
    FN INT,
    Active INT,
    club_member_status VARCHAR(50),
    fashion_news_frequency VARCHAR(50),
    age INT,
    postal_code CHAR(64)
) ENGINE=InnoDB;
""")

0

In [3]:
cur.execute("DROP TABLE IF EXISTS hm_transactions;")
cur.execute("""
CREATE TABLE hm_transactions (
    t_dat DATE,
    customer_id VARCHAR(64),
    article_id BIGINT,
    price DECIMAL(10,5),
    sales_channel_id INT,
    INDEX idx_customer (customer_id),
    INDEX idx_article (article_id)
) ENGINE=InnoDB;
""")

0

In [13]:
article_rows = []
with open("articles.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        article_rows.append((
            row["article_id"],
            row["prod_name"],
            row["product_type_name"],
            row["product_group_name"],
            row["department_name"],
            row["index_name"],
            row["colour_group_name"],
            row["garment_group_name"],
            row["detail_desc"]
        ))

cur.executemany("""
INSERT INTO hm_articles VALUES (
    %s,%s,%s,%s,%s,%s,%s,%s,%s
)
""", article_rows)

105542

In [4]:
def to_int_or_none(v):
    if v is None or v == 'NULL':
        return None
    v = str(v).strip()
    return int(v) if v.isdigit() else None

def to_null(v):
    if v is None:
        return None
    v = str(v).strip()
    return None if v == '' or v == 'NULL' else v

customer_rows = []

with open("customers.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        customer_rows.append((
            row["customer_id"],
            to_int_or_none(row["FN"]),
            to_int_or_none(row["Active"]),
            to_null(row["club_member_status"]),
            to_null(row["fashion_news_frequency"]),
            to_int_or_none(row["age"]),
            to_null(row["postal_code"])
        ))

cur.executemany("""
INSERT INTO hm_customers
(customer_id, FN, Active, club_member_status, fashion_news_frequency, age, postal_code)
VALUES (%s,%s,%s,%s,%s,%s,%s)
""", customer_rows)
conn.commit()

In [ ]:
batch = []
batch_size = 500000

with open("transactions.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        batch.append((
            row["t_dat"],
            row["customer_id"],
            row["article_id"],
            row["price"],
            row["sales_channel_id"]
        ))

        if len(batch) >= batch_size:
            cur.executemany("""
                INSERT INTO hm_transactions
                (t_dat, customer_id, article_id, price, sales_channel_id)
                VALUES (%s,%s,%s,%s,%s)
            """, batch)
            conn.commit()
            print("Inserted", len(batch))
            batch = []


if batch:
    cur.executemany("""
        INSERT INTO hm_transactions
        (t_dat, customer_id, article_id, price, sales_channel_id)
        VALUES (%s,%s,%s,%s,%s)
    """, batch)
    conn.commit()



Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000


In [9]:
cur.execute("SELECT COUNT(*) AS total FROM hm_articles;")
rows = cur.fetchall()
print(rows)



[{'total': 105542}]


In [11]:
cur.execute("SELECT COUNT(*) AS total FROM hm_customers;")
rows = cur.fetchall()
print(rows)



[{'total': 1371980}]


In [7]:
cur.execute("SELECT COUNT(*) AS total FROM hm_transactions;")
rows = cur.fetchall()
print(rows)



[{'total': 31788324}]


In [13]:
# Add the metric columns that duckdb_metrics.py push and score_churn.py fill in

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN customer_lifetime_value DECIMAL(12,2)
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN cohort_month VARCHAR(7)
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN days_since_last_purchase INT
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN total_transactions INT
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN first_purchase_date DATE
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN last_purchase_date DATE
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN churn_risk_score FLOAT
""")

conn.commit()
print("✓ All columns added successfully")

✓ All columns added successfully
